In [ ]:
%pip install plotly pandas


In [ ]:
dbutils.library.restartPython()


In [ ]:
import pandas as pd
import plotly.express as px
df = pd.read_csv('/Volumes/insight/default/titanic/Titanic.csv')
print(f"✅ df loaded — {df.shape[0]} rows x {df.shape[1]} columns")


In [ ]:
# Rename the column in place
df.rename(columns={"2urvived": "Survived"}, inplace=True)

# Verify the column name update
print("Updated columns:", df.columns.tolist())


In [ ]:
fig = px.histogram(
    df,
    x        = "Age",
    title    = "Age Distribution of Titanic Passengers",
    color_discrete_sequence = ["#636EFA"],
    template = "plotly_white"
)

fig.update_layout(
    xaxis_title = "Age",
    yaxis_title = "Count",
)

fig.show()


In [ ]:
survived_counts = df["Survived"].value_counts().reset_index()
survived_counts.columns = ["Survived", "Count"]
survived_counts["Survived"] = survived_counts["Survived"].map({0: "Did Not Survive", 1: "Survived"})

fig = px.bar(
    survived_counts,
    x        = "Survived",
    y        = "Count",
    title    = "Survival Count",
    color    = "Survived",
    color_discrete_map = {
        "Survived"        : "#00CC96",
        "Did Not Survive" : "#EF553B"
    },
    template = "plotly_white"
)

fig.update_layout(
    xaxis_title  = "Outcome",
    yaxis_title  = "Number of Passengers",
    showlegend   = False
)

fig.show()


In [ ]:
fig = px.scatter(
    df,
    x        = "Age",
    y        = "Fare",
    color    = "Survived",
    title    = "Age vs Fare — coloured by Survival",
    color_discrete_map = {0: "#EF553B", 1: "#00CC96"},
    template = "plotly_white",
    opacity  = 0.7
)

fig.update_layout(
    xaxis_title = "Age",
    yaxis_title = "Fare Paid",
)

fig.update_traces(marker=dict(size=6))

fig.show()


In [ ]:
pclass_counts = df["Pclass"].value_counts().reset_index()
pclass_counts.columns = ["Pclass", "Count"]
pclass_counts["Pclass"] = pclass_counts["Pclass"].map({
    1: "1st Class",
    2: "2nd Class",
    3: "3rd Class"
})

fig = px.pie(
    pclass_counts,
    names    = "Pclass",
    values   = "Count",
    title    = "Passenger Class Distribution",
    color_discrete_sequence = ["#636EFA", "#EF553B", "#00CC96"],
    template = "plotly_white"
)

fig.update_traces(textposition="inside", textinfo="percent+label")

fig.show()


In [ ]:
corr = df.select_dtypes(include="number").corr().round(2)

fig = px.imshow(
    corr,
    text_auto = True,
    title     = "Correlation Heatmap",
    color_continuous_scale = "RdBu_r",
    template  = "plotly_white"
)

fig.update_layout(
    width  = 600,
    height = 500
)

fig.show()


In [ ]:
survival = df.groupby("Sex")["Survived"].mean().reset_index()
survival.columns = ["Sex", "Survival Rate"]
survival["Survival Rate"] = survival["Survival Rate"].round(2)
survival["Label"] = survival["Survival Rate"].astype(str)

fig = px.bar(
    survival,
    x        = "Sex",
    y        = "Survival Rate",
    title    = "Survival Rate by Gender",
    color    = "Sex",
    text     = "Label",
    template = "plotly_white",
    color_discrete_sequence = ["#636EFA", "#EF553B"]
)

fig.update_traces(textposition="outside")

fig.update_layout(
    yaxis_title      = "Survival Rate (0 = 0%,  1 = 100%)",
    xaxis_title      = "Gender",
    showlegend       = False,
    yaxis_range      = [0, 1.1]
)

fig.show()

print("\nExact numbers:")
print(survival.to_string(index=False))


In [ ]:
# Register Titanic as a SQL table so we can query it
spark.sql("DROP TABLE IF EXISTS titanic")

spark_df = spark.createDataFrame(df)
spark_df.write.mode("overwrite").saveAsTable("titanic")

print("✅ Titanic table created — ready for SQL queries")
